In [1]:
# Run this cell first (kernel: project .venv — `uv sync --group dev`).
import importlib.util
import json
import os
import re
import sys
from pathlib import Path

import pandas as pd
import requests
from datasets import (
    get_dataset_config_names,
    get_dataset_split_names,
    load_dataset,
)
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display


def _locate_load_env_py() -> Path:
    starts: list[Path] = [Path.cwd().resolve()]
    if "__vsc_ipynb_file__" in globals():
        starts.insert(0, Path(__vsc_ipynb_file__).resolve().parent)
    seen: set[Path] = set()
    for start in starts:
        for candidate in [start, *start.parents]:
            if candidate in seen:
                continue
            seen.add(candidate)
            env_file = candidate / "load_env.py"
            if env_file.is_file():
                return env_file
    raise FileNotFoundError(
        "load_env.py not found. cd to the Romcom-Drift repo root "
        "(or clone it) before running this notebook."
    )


_load_env_path = _locate_load_env_py()
_spec = importlib.util.spec_from_file_location("load_env", _load_env_path)
if _spec is None or _spec.loader is None:
    raise ImportError(f"Cannot load {_load_env_path}")
load_env = importlib.util.module_from_spec(_spec)
sys.modules["load_env"] = load_env
_spec.loader.exec_module(load_env)

PROJECT_DIR = load_env.bootstrap_project()
api = HfApi()

print("PROJECT_DIR:", PROJECT_DIR)
print("Python:", sys.executable)
print("HF_TOKEN configured:", load_env.huggingface_token_configured())

for folder in [
    "data/raw",
    "data/processed",
    "data/metadata",
    "outputs",
    "logs",
]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

PROJECT_DIR: /home/shrys/dev/Romcom-Drift
Python: /home/shrys/dev/Romcom-Drift/.venv/bin/python
HF_TOKEN configured: True


In [2]:
# auditing datasets, repo metadata
api = HfApi()

DATASETS = [
    "Divyanshu/indicxnli",
    "ai4bharat/IndicXParaphrase",
    "ai4bharat/indic-align",
]

dataset_inventory = []

for repo in DATASETS:

    print(f"\nInspecting: {repo}")

    try:
        info = api.dataset_info(
            repo_id=repo,
            files_metadata=False,
        )

        card = info.cardData or {}

        files = [
            file.rfilename
            for file in (info.siblings or [])
        ]

        result = {
            "repository": repo,
            "revision": info.sha,
            "license": card.get("license"),
            "number_of_files": len(files),
            "files": files[:15],
            "status": "ACCESSIBLE",
        }

        dataset_inventory.append(result)

        print(json.dumps(result, indent=2))

    except Exception as e:

        print("ACCESS ERROR:", str(e))

        dataset_inventory.append({
            "repository": repo,
            "status": "FAILED",
            "error": str(e),
        })


Inspecting: Divyanshu/indicxnli
{
  "repository": "Divyanshu/indicxnli",
  "revision": "7092c27872e919f31d0496fb8b9c47bd2cba3f6c",
  "license": [
    "cc0-1.0"
  ],
  "number_of_files": 59,
  "files": [
    ".gitattributes",
    ".gitignore",
    "README.md",
    "backward/dev/xnli_as.json",
    "backward/dev/xnli_bn.json",
    "backward/dev/xnli_gu.json",
    "backward/dev/xnli_hi.json",
    "backward/dev/xnli_kn.json",
    "backward/dev/xnli_ml.json",
    "backward/dev/xnli_mr.json",
    "backward/dev/xnli_or.json",
    "backward/dev/xnli_pa.json",
    "backward/dev/xnli_ta.json",
    "backward/dev/xnli_te.json",
    "backward/test/xnli_as.json"
  ],
  "status": "ACCESSIBLE"
}

Inspecting: ai4bharat/IndicXParaphrase
{
  "repository": "ai4bharat/IndicXParaphrase",
  "revision": "181031f9e69c465e0c72fc84fd33b66114adb3e8",
  "license": null,
  "number_of_files": 12,
  "files": [
    ".gitattributes",
    "IndicXParaphrase.py",
    "data/as.tsv",
    "data/bn.tsv",
    "data/gu.tsv",
  

In [3]:
DAKSHINA_URL = (
    "https://storage.googleapis.com/gresearch/dakshina/"
    "dakshina_dataset_v1.0.tar"
)

response = requests.head(
    DAKSHINA_URL,
    allow_redirects=True,
    timeout=30,
)

print("HTTP status:", response.status_code)

size = response.headers.get("Content-Length")

if size:
    print("Archive size:", round(int(size) / 1024**2, 2), "MB")
else:
    print("Archive size: Not reported")

print("Content type:", response.headers.get("Content-Type"))

HTTP status: 200
Archive size: 1915.3 MB
Content type: application/x-tar


In [4]:
audit_path = (
    PROJECT_DIR / "data/metadata/dataset_inventory.json"
)

audit_path.write_text(
    json.dumps(dataset_inventory, indent=2),
    encoding="utf-8",
)

print("Inventory saved:", audit_path)

Inventory saved: /home/shrys/dev/Romcom-Drift/data/metadata/dataset_inventory.json


In [5]:
# Dataset probe cells use imports and `api` from the setup cell above.

In [7]:
repo = "Divyanshu/indicxnli"

info = api.dataset_info(repo)

layout = sorted(
    {
        sibling.rfilename.split("/")[0]
        for sibling in (info.siblings or [])
        if "/" in sibling.rfilename
    }
)
langs = sorted(
    {
        match.group(1)
        for sibling in (info.siblings or [])
        for match in [re.search(r"xnli_(\w+)\.json", sibling.rfilename)]
        if match
    }
)

print("TOP-LEVEL LAYOUT:", layout)
print("LANGUAGE CODES (from filenames):", langs)

print("\nHINDI TRAIN FILES:")
for sibling in info.siblings or []:
    name = sibling.rfilename
    if "hi" in name.lower() and "train" in name.lower():
        print(name)

TOP-LEVEL LAYOUT: ['backward', 'forward']
LANGUAGE CODES (from filenames): ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'or', 'pa', 'ta', 'te']

HINDI TRAIN FILES:
forward/train/xnli_hi.json


In [ ]:
repo = "ai4bharat/IndicXParaphrase"

file_path = hf_hub_download(
    repo_id=repo,
    filename="data/hi.tsv",
    repo_type="dataset"
)

df = pd.read_csv(file_path, sep="\t")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head(5))

print("\nLabel distribution:")
print(df["label"].value_counts(dropna=False))

hi.tsv:   0%|          | 0.00/1.21M [00:00<?, ?B/s]

Shape: (2002, 4)

Columns:
['en', 'sentence1', 'sentence2', 'label']


,en,sentence1,sentence2,label
0,"on receiving the information, the police reach...",सूचना मिलते ही पुलिस मौके पर पहुंची और घायलों ...,पुलिस मौके पर सूचना मिलते ही पहुंची और घायलों ...,1
1,"on receiving the information, the police reach...",सूचना मिलते ही पुलिस मौके पर पहुंची और घायलों ...,सूचना मिलते ही घायलों को स्थानीय अस्पताल में भ...,0
2,he won the filmfare award for best supporting ...,इस फिल्म में उनके अभिनय के लिए उन्हें सर्वश्रे...,सर्वश्रेष्ठ सहायक अभिनेता का फिल्मफेयर पुरस्का...,1
3,he won the filmfare award for best supporting ...,इस फिल्म में उनके अभिनय के लिए उन्हें सर्वश्रे...,उनके अभिनय के लिए उन्हें सर्वश्रेष्ठ सहायक अभि...,0
4,chandigarh: punjab kala parishad would honour ...,चंडीगढ़ः पंजाब कला परिषद प्रख्यात पंजाबी साहित...,चंडीगढ़ः प्रख्यात पंजाबी साहित्यकार जसवंत सिं...,1



Label distribution:
label
1    1001
0    1001
Name: count, dtype: int64


In [ ]:
repo = "ai4bharat/indic-align"

configs = get_dataset_config_names(repo)

print("AVAILABLE CONFIGURATIONS:")
print(configs)

toxic_config = next(
    (c for c in configs if "toxic" in c.lower() and "matrix" in c.lower()),
    None
)

assert toxic_config is not None, "Toxic Matrix config not found"

splits = get_dataset_split_names(
    repo,
    config_name=toxic_config
)

print("\nSplits:", splits)

ds = load_dataset(
    repo,
    name=toxic_config,
    split=splits[0],
    streaming=True
)

print("\nFeatures:", ds.features)

for i, row in enumerate(ds.take(3)):
    print(f"\nEXAMPLE {i+1}")

    for key, value in row.items():
        print(f"{key}: {str(value)[:500]}")

README.md:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

AVAILABLE CONFIGURATIONS:
['Indic_ShareLlama', 'Dolly_T', 'OpenAssistant_T', 'WikiHow', 'IndoWordNet', 'Anudesh', 'Wiki_Conv', 'Wiki_Chat', 'HHRLHF_T', 'Toxic_Matrix']

Splits: ['train']

Features: {'doc_id': Value('string'), 'num_turns': Value('float64'), 'eng_Latn': List(List(Value('string'))), 'asm_Beng': List(List(Value('string'))), 'asm_Latn': List(List(Value('string'))), 'ben_Beng': List(List(Value('string'))), 'ben_Latn': List(List(Value('string'))), 'guj_Gujr': List(List(Value('string'))), 'hin_Deva': List(List(Value('string'))), 'hin_Latn': List(List(Value('string'))), 'kan_Knda': List(List(Value('string'))), 'kan_Latn': List(List(Value('string'))), 'mal_Mlym': List(List(Value('string'))), 'mal_Latn': List(List(Value('string'))), 'mar_Deva': List(List(Value('string'))), 'mar_Latn': List(List(Value('string'))), 'npi_Deva': List(List(Value('string'))), 'npi_Latn': List(List(Value('string'))), 'ory_Orya': List(List(Value('string'))), 'ory_Latn': List(List(Value('string'))), 'pan_

In [ ]:
repo = "LingoIITGN/COMI-LINGUA"

configs = get_dataset_config_names(repo)

print("CONFIGURATIONS:")
print(configs)

for config_name in configs:

    print(f"\n{'='*50}")
    print("CONFIG:", config_name)

    splits = get_dataset_split_names(
        repo,
        config_name=config_name
    )

    print("Splits:", splits)

    ds = load_dataset(
        repo,
        name=config_name,
        split="train",
        streaming=True
    )

    print("Features:", ds.features)

    for i, row in enumerate(ds.take(2)):

        print(f"\nSample {i+1}")

        for key, value in row.items():
            print(f"{key}: {str(value)[:400]}")

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

CONFIGURATIONS:
['LID', 'MLI', 'POS', 'NER', 'TN', 'MT']

CONFIG: LID
Splits: ['train', 'test']
Features: None

Sample 1
Sentences: इस स्पोकन ट्यूटोरियल में हम usb driver संस्थापित करने के लिए कंप्यूटर और sbhs के बीच फिज़िकल कम्युनिकेशन सेटअप करना सीखेंगे|
Predicted tags: [{'key': 'इस', 'value': 'hi', 'index': 0}, {'key': 'स्पोकन', 'value': 'hi', 'index': 1}, {'key': 'ट्यूटोरियल', 'value': 'hi', 'index': 2}, {'key': 'में', 'value': 'hi', 'index': 3}, {'key': 'हम', 'value': 'hi', 'index': 4}, {'key': 'usb', 'value': 'en', 'index': 5}, {'key': 'driver', 'value': 'en', 'index': 6}, {'key': 'संस्थापित', 'value': 'hi', 'index': 7}, {'key': 'करने', 'value': 'hi', 'index'
Annotated by: Annotator 1: [{'key': 'इस', 'value': 'hi', 'index': 0}, {'key': 'स्पोकन', 'value': 'en', 'index': 1}, {'key': 'ट्यूटोरियल', 'value': 'en', 'index': 2}, {'key': 'में', 'value': 'hi', 'index': 3}, {'key': 'हम', 'value': 'hi', 'index': 4}, {'key': 'usb', 'value': 'en', 'index': 5}, {'key': 'driver', 'value': 'en',

In [8]:
from datasets import load_dataset
import pandas as pd

repo = "LingoIITGN/COMI-LINGUA"

mli = load_dataset(
    repo,
    name="MLI",
    split="train"
)

print("Rows:", len(mli))
print("\nColumns:")
print(mli.column_names)

print("\nFeatures:")
print(mli.features)

for i in range(5):
    row = mli[i]

    print(f"\nSAMPLE {i+1}")

    for key, value in row.items():
        if isinstance(value, str):
            print(f"{key}: {value[:250]}")
        else:
            print(f"{key}: {str(value)[:250]}")

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

MLI_train.csv:   0%|          | 0.00/5.55M [00:00<?, ?B/s]

MLI_test.csv:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Rows: 26225

Columns:
['Sentences', 'Annotated by: Annotator 1', 'Annotated by: Annotator 2', 'Annotated by: Annotator 3']

Features:
{'Sentences': Value('large_string'), 'Annotated by: Annotator 1': Value('large_string'), 'Annotated by: Annotator 2': Value('large_string'), 'Annotated by: Annotator 3': Value('large_string')}

SAMPLE 1
Sentences: आईपीएल 2021 (IPL 2021) के 43वें मुकाबले में आरसीबी ने राजस्थान रॉयल्स (RCB vs RR) को 7 विकेट से हरा दिया।
Annotated by: Annotator 1: hi
Annotated by: Annotator 2: hi
Annotated by: Annotator 3: hi

SAMPLE 2
Sentences: पराली का समाधान करने के लिए वीरेन्द्र जी ने, पुआल की गांठ बनाने वाली straw baler मशीन खरीदी।
Annotated by: Annotator 1: hi
Annotated by: Annotator 2: hi
Annotated by: Annotator 3: hi

SAMPLE 3
Sentences: Delhi is capital u fool Ã°Å¸Ëœâ€š Capitol me hamesa rulling party ka hi raaj chalta he jo ki bjp keÃ°Å¸Ëœâ€š
Annotated by: Annotator 1: hi
Annotated by: Annotator 2: hi
Annotated by: Annotator 3: hi

SAMPLE 4
Sentences: Mera dil ka

In [9]:
roman_fields = [
    col for col in mli.column_names
    if "rh" in col.lower()
]

print("Romanized Hindi fields:", roman_fields)

for field in roman_fields:

    values = [
        x for x in mli[field]
        if isinstance(x, str) and x.strip()
    ]

    print(
        field,
        "| Non-empty:", len(values),
        "| Unique:", len(set(values))
    )

    print("Examples:")
    for x in values[:5]:
        print("-", x)

Romanized Hindi fields: []


In [10]:
tn = load_dataset(
    repo,
    name="TN",
    split="train"
)

print("Rows:", len(tn))
print("Columns:", tn.column_names)
print("Features:", tn.features)

for i in range(5):
    print("\n", tn[i])

TN_train.csv: reconstructing file:   0%|          |  0.00B / 12.1MB            

TN_train.csv: downloading bytes:           |  0.00B            

TN_test.csv:   0%|          | 0.00/2.86M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Rows: 24547
Columns: ['Sentences', 'Predicted Tags', 'Annotated by: Annotator 1', 'Annotated by: Annotator 2', 'Annotated by: Annotator 3']
Features: {'Sentences': Value('large_string'), 'Predicted Tags': Value('large_string'), 'Annotated by: Annotator 1': Value('large_string'), 'Annotated by: Annotator 2': Value('large_string'), 'Annotated by: Annotator 3': Value('large_string')}

 {'Sentences': 'Thank u so much... Aapke madad se ham bahut kuch sikhe hai... Aap hamesa video banate rahiye... Again thanks a lot', 'Predicted Tags': 'Thank you so much. Aapke madad se hum bahut kuch sikhe hain. Aap hamesha video banate rahiye. Again, thanks a lot.', 'Annotated by: Annotator 1': 'Thank you so much. Aapki madad se hum bahut kuch seekhe hain. Aap hamesha video banate rahiye. Again, thanks a lot.', 'Annotated by: Annotator 2': 'Thank you so much. Aapke madad se hum bahut kuch sikhe hai aap hamesha video banate rahiye. Again, thanks a lot.', 'Annotated by: Annotator 3': 'Thank you so much. Aapk

In [11]:
from datasets import load_dataset
import pandas as pd

mli = load_dataset(
    "LingoIITGN/COMI-LINGUA",
    "MLI",
    split="train"
)

df = mli.to_pandas()

annotator_cols = [
    "Annotated by: Annotator 1",
    "Annotated by: Annotator 2",
    "Annotated by: Annotator 3"
]

# Missing values
print("Missing values:")
print(df[["Sentences"] + annotator_cols].isna().sum())

# Annotator disagreement
df["unique_annotations"] = df[annotator_cols].nunique(axis=1)

print("\nAnnotation agreement:")
print(df["unique_annotations"].value_counts().sort_index())

# Percentage of examples with disagreement
print(
    "\nDisagreement rate:",
    (df["unique_annotations"] > 1).mean() * 100,
    "%"
)

# Show examples with maximum disagreement
examples = df[df["unique_annotations"] == 3]

for _, row in examples.head(10).iterrows():
    print("\nSOURCE:", row["Sentences"])
    for col in annotator_cols:
        print(col, ":", row[col])

Missing values:
Sentences                    0
Annotated by: Annotator 1    0
Annotated by: Annotator 2    4
Annotated by: Annotator 3    2
dtype: int64

Annotation agreement:
unique_annotations
1    25778
2      447
Name: count, dtype: int64

Disagreement rate: 1.7044804575786463 %


In [12]:
tn = load_dataset(
    "LingoIITGN/COMI-LINGUA",
    "TN",
    split="train"
)

tn_df = tn.to_pandas()

print("Rows:", len(tn_df))
print("Columns:", tn_df.columns.tolist())
print("\nMissing values:")
print(tn_df.isna().sum())

print("\nSample:")
for _, row in tn_df.head(10).iterrows():
    for col in tn_df.columns:
        print(f"{col}: {row[col]}")
    print("-" * 80)

Rows: 24547
Columns: ['Sentences', 'Predicted Tags', 'Annotated by: Annotator 1', 'Annotated by: Annotator 2', 'Annotated by: Annotator 3']

Missing values:
Sentences                    0
Predicted Tags               0
Annotated by: Annotator 1    0
Annotated by: Annotator 2    0
Annotated by: Annotator 3    0
dtype: int64

Sample:
Sentences: Thank u so much... Aapke madad se ham bahut kuch sikhe hai... Aap hamesa video banate rahiye... Again thanks a lot
Predicted Tags: Thank you so much. Aapke madad se hum bahut kuch sikhe hain. Aap hamesha video banate rahiye. Again, thanks a lot.
Annotated by: Annotator 1: Thank you so much. Aapki madad se hum bahut kuch seekhe hain. Aap hamesha video banate rahiye. Again, thanks a lot.
Annotated by: Annotator 2: Thank you so much. Aapke madad se hum bahut kuch sikhe hai aap hamesha video banate rahiye. Again, thanks a lot.
Annotated by: Annotator 3: Thank you so much. Aapke madad se hum bahut kuch seekhe hain. Aap hamesha video banate rahiye. Agai

In [13]:
import re

english_pattern = re.compile(r"\b[a-zA-Z]+\b")

def english_token_ratio(text):
    tokens = str(text).split()
    if not tokens:
        return 0
    english = sum(
        bool(english_pattern.fullmatch(token.strip(".,!?;:()[]{}")))
        for token in tokens
    )
    return english / len(tokens)

df["english_ratio"] = df["Sentences"].apply(english_token_ratio)

print(df["english_ratio"].describe())

print("\nEnglish token ratio buckets:")
print(
    pd.cut(
        df["english_ratio"],
        bins=[-0.01, 0.05, 0.20, 0.40, 1.0],
        labels=["Very low", "Low", "Moderate", "High"]
    ).value_counts()
)

count    26225.000000
mean         0.355831
std          0.334608
min          0.000000
25%          0.105263
50%          0.190476
75%          0.714286
max          1.000000
Name: english_ratio, dtype: float64

English token ratio buckets:
english_ratio
Low         10229
High         8493
Moderate     4066
Very low     3437
Name: count, dtype: int64


In [14]:
import re
import pandas as pd

annotator_cols = [
    "Annotated by: Annotator 1",
    "Annotated by: Annotator 2",
    "Annotated by: Annotator 3"
]

# What labels are annotators assigning?
for col in annotator_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).head(20))

# Approximate script composition
def script_counts(text):
    text = str(text)
    devanagari = sum('\u0900' <= c <= '\u097F' for c in text)
    latin = sum(('a' <= c.lower() <= 'z') for c in text)
    other = sum(c.isalpha() and not (
        '\u0900' <= c <= '\u097F' or
        'a' <= c.lower() <= 'z'
    ) for c in text)

    total = devanagari + latin + other
    return pd.Series({
        "devanagari_ratio": devanagari / total if total else 0,
        "latin_ratio": latin / total if total else 0,
        "other_ratio": other / total if total else 0
    })

script_df = df["Sentences"].apply(script_counts)

print("\nScript composition:")
print(script_df.describe())

print("\nExamples with substantial Latin script:")
for text in df.loc[script_df["latin_ratio"] > 0.3, "Sentences"].head(10):
    print("-", text)


Annotated by: Annotator 1
Annotated by: Annotator 1
hi    25166
en     1059
Name: count, dtype: int64

Annotated by: Annotator 2
Annotated by: Annotator 2
hi     25266
en       955
NaN        4
Name: count, dtype: int64

Annotated by: Annotator 3
Annotated by: Annotator 3
hi     25095
en      1128
NaN        2
Name: count, dtype: int64

Script composition:
       devanagari_ratio   latin_ratio   other_ratio
count      26225.000000  26225.000000  26225.000000
mean           0.544715      0.454337      0.000948
std            0.381238      0.380176      0.013558
min            0.000000      0.000000      0.000000
25%            0.000000      0.150442      0.000000
50%            0.710843      0.289062      0.000000
75%            0.849462      1.000000      0.000000
max            1.000000      1.000000      0.681818

Examples with substantial Latin script:
- Delhi is capital u fool Ã°Å¸Ëœâ€š Capitol me hamesa rulling party ka hi raaj chalta he jo ki bjp keÃ°Å¸Ëœâ€š
- Mera dil kal schoo

In [15]:
import pandas as pd
from difflib import SequenceMatcher

tn_annotators = [
    "Annotated by: Annotator 1",
    "Annotated by: Annotator 2",
    "Annotated by: Annotator 3"
]

# Annotation agreement
tn_df["unique_annotations"] = tn_df[tn_annotators].nunique(axis=1)

print("Annotation agreement:")
print(tn_df["unique_annotations"].value_counts().sort_index())

print(
    "\nRows with annotation disagreement:",
    (tn_df["unique_annotations"] > 1).mean() * 100,
    "%"
)

# Agreement between predicted output and human annotations
for col in tn_annotators:
    exact_match = (
        tn_df["Predicted Tags"].str.strip().str.lower()
        == tn_df[col].str.strip().str.lower()
    ).mean()

    print(f"{col} exact match with prediction: {exact_match:.2%}")

# How much does normalization change the original text?
def similarity(a, b):
    return SequenceMatcher(
        None,
        str(a).lower().strip(),
        str(b).lower().strip()
    ).ratio()

tn_df["source_to_annotation_similarity"] = tn_df.apply(
    lambda row: similarity(
        row["Sentences"],
        row["Annotated by: Annotator 1"]
    ),
    axis=1
)

print("\nSource-to-annotation similarity:")
print(tn_df["source_to_annotation_similarity"].describe())

print("\nExamples with substantial changes:")
for _, row in tn_df.nsmallest(
    10, "source_to_annotation_similarity"
).iterrows():
    print("\nSOURCE:", row["Sentences"])
    print("OUTPUT:", row["Annotated by: Annotator 1"])
    print("SIMILARITY:", row["source_to_annotation_similarity"])

Annotation agreement:
unique_annotations
1      258
2     3094
3    21195
Name: count, dtype: int64

Rows with annotation disagreement: 98.94895506579215 %
Annotated by: Annotator 1 exact match with prediction: 17.87%
Annotated by: Annotator 2 exact match with prediction: 7.93%
Annotated by: Annotator 3 exact match with prediction: 6.52%

Source-to-annotation similarity:
count    24547.000000
mean         0.928572
std          0.059056
min          0.062338
25%          0.907801
50%          0.938053
75%          0.962963
max          1.000000
Name: source_to_annotation_similarity, dtype: float64

Examples with substantial changes:

SOURCE: Jis bhi admi ne ye kam kiya hai....usko jel me dalna chahiye turant Normal family me beti ke liye pita ke bad.... pti hota....yhi sikh di jati hai....or kisi ki beti ke sath itni gandgi
OUTPUT: Jis bhi aadmi ne yeh kaam kiya hai usko jail mein daalna chahiye turant. Normal family mein beti ke liye pehle pita hota, phir pati hota. Yahi sikh di jaati 

In [16]:
from datasets import load_dataset

mt = load_dataset(
    "LingoIITGN/COMI-LINGUA",
    "MT"
)

print(mt)
print("\nTrain rows:", len(mt["train"]))
print("Test rows:", len(mt["test"]))

print("\nFeatures:")
print(mt["train"].features)

print("\nColumns:")
print(mt["train"].column_names)

for i in range(10):
    print(f"\n--- SAMPLE {i+1} ---")
    for key, value in mt["train"][i].items():
        print(f"{key}: {value}")

MT_train.csv: reconstructing file:   0%|          |  0.00B / 61.0MB            

MT_train.csv: downloading bytes:           |  0.00B            

MT_test.csv: reconstructing file:   0%|          |  0.00B / 11.6MB            

MT_test.csv: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Sentences', 'Predicted_en_translation', 'Predicted_RH_translation', 'Predicted_DH_translation', 'Annotator_1_en_translation', 'Annotator_1_RH_translation', 'Annotator_1_DH_translation', 'Annotator_2_en_translation', 'annotator2_RH_translation', 'Annotator_2_DH_translation', 'Annotator_3_en_translation', 'annotator3_RH_translation', 'Annotator_3_DH_translation'],
        num_rows: 24692
    })
    test: Dataset({
        features: ['Sentences', 'Predicted_en_translation', 'Predicted_RH_translation', 'Predicted_DH_translation', 'Annotator_1_en_translation', 'Annotator_1_RH_translation', 'Annotator_1_DH_translation', 'Annotator_2_en_translation', 'annotator2_RH_translation', 'Annotator_2_DH_translation', 'Annotator_3_en_translation', 'annotator3_RH_translation', 'Annotator_3_DH_translation'],
        num_rows: 5000
    })
})

Train rows: 24692
Test rows: 5000

Features:
{'Sentences': Value('large_string'), 'Predicted_en_translation': 

In [17]:
import pandas as pd

mt_df = mt["train"].to_pandas()

print("Shape:", mt_df.shape)
print("\nMissing values:")
print(mt_df.isna().sum())

print("\nExact duplicate rows:", mt_df.duplicated().sum())

# Show whether the same source text occurs more than once
for col in mt_df.columns:
    print(
        f"\nDuplicate values in {col}:",
        mt_df[col].duplicated().sum()
    )

Shape: (24692, 13)

Missing values:
Sentences                     0
Predicted_en_translation      0
Predicted_RH_translation      0
Predicted_DH_translation      0
Annotator_1_en_translation    0
Annotator_1_RH_translation    0
Annotator_1_DH_translation    0
Annotator_2_en_translation    0
annotator2_RH_translation     0
Annotator_2_DH_translation    0
Annotator_3_en_translation    0
annotator3_RH_translation     0
Annotator_3_DH_translation    0
dtype: int64

Exact duplicate rows: 0

Duplicate values in Sentences: 0

Duplicate values in Predicted_en_translation: 22

Duplicate values in Predicted_RH_translation: 22

Duplicate values in Predicted_DH_translation: 22

Duplicate values in Annotator_1_en_translation: 21

Duplicate values in Annotator_1_RH_translation: 12

Duplicate values in Annotator_1_DH_translation: 17

Duplicate values in Annotator_2_en_translation: 21

Duplicate values in annotator2_RH_translation: 17

Duplicate values in Annotator_2_DH_translation: 23

Duplicate valu

In [19]:
import pandas as pd

rh_cols = [
    "Predicted_RH_translation",
    "Annotator_1_RH_translation",
    "annotator2_RH_translation",
    "annotator3_RH_translation"
]

# Number of distinct Romanized Hindi outputs per source
mt_df["rh_unique_count"] = mt_df[rh_cols].nunique(axis=1)

print("Distinct RH outputs per source:")
print(mt_df["rh_unique_count"].value_counts().sort_index())

print(
    "\nRows with at least 2 different RH outputs:",
    (mt_df["rh_unique_count"] >= 2).mean() * 100,
    "%"
)

print(
    "Rows where all 4 RH outputs differ:",
    (mt_df["rh_unique_count"] == 4).sum()
)

# Show examples with maximum variation
for _, row in mt_df[mt_df["rh_unique_count"] == 4].head(10).iterrows():
    print("\nSOURCE:", row["Sentences"])
    for col in rh_cols:
        print(f"{col}: {row[col]}")

Distinct RH outputs per source:
rh_unique_count
1    16473
2     6790
3     1097
4      332
Name: count, dtype: int64

Rows with at least 2 different RH outputs: 33.286084561801395 %
Rows where all 4 RH outputs differ: 332

SOURCE: साल 2019 के लोकसभा चुनाव (Lok Sabha Election 2019) में कुशवाहा को महागठबंधन (Grand Alliance) मेंं मिली दोनों सीटों पर शिकस्‍त मिली।
Predicted_RH_translation: 2019 के लोकसभा चुनाव में, कुशवाहा को ग्रैंड अलायंस में उन्हें आवंटित दोनों सीटों पर हार का सामना करना पड़ा।
Annotator_1_RH_translation: 2019 ke loksabha chunav mein kushwaha ko mahagathbandhan mein mili dono sito par sikasht mili.
annotator2_RH_translation: 2019ke loksabha chunav me kushwaha ko mahagadhbandhan me mili dono seeto par sikast mili.
annotator3_RH_translation: 2019 ke loksabha chunav me Kushwaha ko mahagathbandhan me mili dono seaton par haar mili.

SOURCE: इस संग्रहालय को अमेरिकी प्राकृतिक इतिहास संग्रहालय (American Museum of Natural History) या स्मिथसोनियन संग्रहालय (Smithsonian Museum) के

In [20]:
train_sources = set(mt["train"]["Sentences"])
test_sources = set(mt["test"]["Sentences"])

overlap = train_sources.intersection(test_sources)

print("Train sources:", len(train_sources))
print("Test sources:", len(test_sources))
print("Overlapping sources:", len(overlap))
print(
    "Test overlap percentage:",
    len(overlap) / len(test_sources) * 100
)

Train sources: 24692
Test sources: 5000
Overlapping sources: 0
Test overlap percentage: 0.0


In [ ]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(
        None,
        str(a).lower().strip(),
        str(b).lower().strip()
    ).ratio()

comparison_pairs = [
    ("Annotator_1_RH_translation", "annotator2_RH_translation"),
    ("Annotator_1_RH_translation", "annotator3_RH_translation"),
    ("annotator2_RH_translation", "annotator3_RH_translation"),
    ("Predicted_RH_translation", "Annotator_1_RH_translation")
]

for a, b in comparison_pairs:
    scores = [
        similarity(x, y)
        for x, y in zip(mt_df[a], mt_df[b])
    ]

    print(f"\n{a} vs {b}")
    print(pd.Series(scores).describe())

In [21]:
import pandas as pd
import re

rh_annotator_cols = [
    "Annotator_1_RH_translation",
    "annotator2_RH_translation",
    "annotator3_RH_translation"
]

def script_profile(text):
    text = str(text)
    devanagari = len(re.findall(r"[\u0900-\u097F]", text))
    latin = len(re.findall(r"[A-Za-z]", text))
    total = devanagari + latin

    return {
        "devanagari_ratio": devanagari / total if total else 0,
        "latin_ratio": latin / total if total else 0
    }

# Distinct annotator outputs per source
annotator_unique = mt_df[rh_annotator_cols].nunique(axis=1)

print("Distinct annotator RH outputs per source:")
print(annotator_unique.value_counts().sort_index())

# Script composition per column
for col in rh_annotator_cols + ["Predicted_RH_translation"]:
    profiles = mt_df[col].apply(script_profile)
    print(f"\n{col}")
    print("Mean Latin ratio:", profiles.apply(lambda x: x["latin_ratio"]).mean())
    print("Mean Devanagari ratio:", profiles.apply(lambda x: x["devanagari_ratio"]).mean())

# Show examples where all three annotators differ
examples = mt_df[annotator_unique == 3]

for idx, row in examples.head(5).iterrows():
    print("\nSOURCE:", row["Sentences"])
    for col in rh_annotator_cols:
        print(f"{col}:", row[col])

Distinct annotator RH outputs per source:
1    16476
2     6795
3     1421
Name: count, dtype: int64

Annotator_1_RH_translation
Mean Latin ratio: 0.9990490862388535
Mean Devanagari ratio: 0.0009509137611466921

annotator2_RH_translation
Mean Latin ratio: 0.9991843009106456
Mean Devanagari ratio: 0.0008156990893544759

annotator3_RH_translation
Mean Latin ratio: 0.9996304705181094
Mean Devanagari ratio: 0.00036952948189061254

Predicted_RH_translation
Mean Latin ratio: 0.9863974421192422
Mean Devanagari ratio: 0.01360255788075792

SOURCE: ब्रिक्स बैंक से जुड़ने से पहले कामत देश की दूसरी सबसे बड़ी आईटी कंपनी Infosys के चेयरमैन और ICICI Bank के नॉन-एक्जीक्यूटिव चेयरमैन थे।
Annotator_1_RH_translation: ब्रिक्स बैंक में शामिल होने से पहले, कमाथ देश की दूसरी सबसे बड़ी आईटी कंपनी इंफोसिस और आईसीआईसीआई बैंक के गैर-कार्यकारी अध्यक्ष थे।
annotator2_RH_translation: BRICS bank se jurne se pehle kamat desh ki dusri sabse bari IT company Infosys ke chairman aur ICICI bank ke non executive chairman t

In [22]:
import re
import pandas as pd

rh_cols = [
    "Annotator_1_RH_translation",
    "annotator2_RH_translation",
    "annotator3_RH_translation",
    "Predicted_RH_translation"
]

def classify_script(text):
    text = str(text)

    devanagari = len(re.findall(r"[\u0900-\u097F]", text))
    latin = len(re.findall(r"[A-Za-z]", text))

    total = devanagari + latin

    if total == 0:
        return "other"
    if latin / total >= 0.8:
        return "latin"
    if devanagari / total >= 0.8:
        return "devanagari"
    return "mixed"

for col in rh_cols:
    print(f"\n{col}")
    print(mt_df[col].apply(classify_script).value_counts())

# Keep only rows where all three annotators are predominantly Latin-script
script_labels = pd.DataFrame({
    col: mt_df[col].apply(classify_script)
    for col in rh_annotator_cols
})

all_latin = (script_labels == "latin").all(axis=1)

print("\nRows where all 3 annotators are Latin-dominant:", all_latin.sum())
print("Percentage:", round(all_latin.mean() * 100, 2))

latin_rows = mt_df.loc[all_latin, rh_annotator_cols]

print("\nDistinct outputs among Latin-only rows:")
print(latin_rows.nunique(axis=1).value_counts().sort_index())


Annotator_1_RH_translation
Annotator_1_RH_translation
latin         24667
devanagari       21
mixed             4
Name: count, dtype: int64

annotator2_RH_translation
annotator2_RH_translation
latin         24672
devanagari       19
mixed             1
Name: count, dtype: int64

annotator3_RH_translation
annotator3_RH_translation
latin         24682
devanagari        8
mixed             2
Name: count, dtype: int64

Predicted_RH_translation
Predicted_RH_translation
latin         24347
devanagari      330
mixed            15
Name: count, dtype: int64

Rows where all 3 annotators are Latin-dominant: 24642
Percentage: 99.8

Distinct outputs among Latin-only rows:
1    16475
2     6792
3     1375
Name: count, dtype: int64


In [23]:
from difflib import SequenceMatcher
from itertools import combinations
import numpy as np

pairs = list(combinations(rh_annotator_cols, 2))

for col_a, col_b in pairs:
    similarities = [
        SequenceMatcher(
            None,
            str(a).lower().strip(),
            str(b).lower().strip()
        ).ratio()
        for a, b in zip(mt_df[col_a], mt_df[col_b])
    ]

    print(f"\n{col_a} vs {col_b}")
    print("Mean:", round(np.mean(similarities), 4))
    print("Median:", round(np.median(similarities), 4))
    print("10th percentile:", round(np.percentile(similarities, 10), 4))
    print("Below 0.8:", sum(s < 0.8 for s in similarities))


Annotator_1_RH_translation vs annotator2_RH_translation
Mean: 0.9946
Median: 1.0
10th percentile: 1.0
Below 0.8: 120

Annotator_1_RH_translation vs annotator3_RH_translation
Mean: 0.9797
Median: 1.0
10th percentile: 0.9373
Below 0.8: 371

annotator2_RH_translation vs annotator3_RH_translation
Mean: 0.9789
Median: 1.0
10th percentile: 0.9343
Below 0.8: 410


In [24]:
train_sources = set(mt["train"]["Sentences"])
test_sources = set(mt["test"]["Sentences"])

overlap = train_sources.intersection(test_sources)

print("Overlap:", len(overlap))
print("Test overlap percentage:", round(len(overlap) / len(test_sources) * 100, 4))

Overlap: 0
Test overlap percentage: 0.0
